In [ ]:
IS_TRAIN = False          
IS_INFERENCE = True 

In [ ]:
 !pip install numpy==1.26.4 --force-reinstall -q

In [ ]:
from pathlib import Path
WORKING = Path("/kaggle/working/")
INPUT = Path ("/kaggle/input/datasets/maituananh511/fine-tune-vintern1b")

In [ ]:
!pip install -q transformers==4.44.* --force-reinstall

In [ ]:
# !pip uninstall numpy -y

In [ ]:
!pip install -q packaging
!pip install -q ninja
!pip install -q flash_attn
!pip install -q datasets
!pip install -q timm
!pip install -q einops
!pip install -q peft
!pip install -q deepspeed
!pip install -q bitsandbytes
!pip install -q decord
!pip install -q gdown

In [ ]:
!git clone https://github.com/5CD-AI/Vintern.git

%cd Vintern

In [ ]:
from datasets import load_from_disk

vi_chart_dataset = load_from_disk("/kaggle/input/datasets/maituananh511/dataset-chart-vqa/vi_chart_dataset")

print(vi_chart_dataset)

In [ ]:
from datasets import load_from_disk, concatenate_datasets
import json
from pathlib import Path
from PIL import Image

In [ ]:
VIETNAMESE_DATA_PATH = Path("/kaggle/input/datasets/maituananh511/data-vietnamese/Data Vietnamese")
VIETNAMESE_IMAGES_PATH = VIETNAMESE_DATA_PATH / "images"
VIETNAMESE_JSONL_PATH = VIETNAMESE_DATA_PATH / "viet_chart_vqa.jsonl"

vietnamese_records = []
with open(VIETNAMESE_JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            vietnamese_records.append(json.loads(line))

print(f"Vietnamese dataset: {len(vietnamese_records)} records loaded")


In [ ]:
from collections import Counter

key_combos = Counter()
bad_records = []
for idx, record in enumerate(vietnamese_records):
    for turn in record['conversations']:
        keys = tuple(sorted(turn.keys())) if isinstance(turn, dict) else ('NOT_DICT', type(turn).__name__)
        key_combos[keys] += 1
        if keys != ('content', 'role'):
            bad_records.append((idx, keys, repr(turn)[:100]))

print('=== Key combinations found in all turns ===')
for combo, count in key_combos.most_common():
    print(f'  {combo}: {count} turns')

print(f'\n=== Bad records (non role/content): {len(bad_records)} ===')
for idx, keys, val in bad_records[:10]:
    print(f'  record[{idx}]: keys={keys}, value={val}')

In [ ]:
from datasets import Dataset
import pyarrow as pa

def normalize_turn(turn):
    if isinstance(turn, str):
        return {'role': 'assistant', 'content': turn}
    if isinstance(turn, dict):
        role = turn.get('role') or turn.get('from', '')
        if role in ('human', 'user'): role = 'user'
        elif role in ('gpt', 'assistant'): role = 'assistant'
        for rk in ('assistant', 'user', 'human', 'gpt'):
            if rk in turn and 'content' not in turn and 'role' not in turn:
                role = 'assistant' if rk in ('assistant', 'gpt') else 'user'
                return {'role': role, 'content': str(turn[rk])}
        content = str(turn.get('content') or turn.get('value', ''))
        return {'role': role, 'content': content}
    return {'role': 'assistant', 'content': str(turn)}

def align_conversations_schema(dataset, reference_dataset):
    ref_conv_type = reference_dataset.data.schema.field('conversations').type
    ref_struct_type = ref_conv_type.value_type
    field_order = [ref_struct_type.field(i).name for i in range(ref_struct_type.num_fields)]
    pa_table = dataset.data.table
    conv_arr = pa_table.column('conversations').combine_chunks()
    struct_arr = conv_arr.values
    arrays = [struct_arr.field(f) for f in field_order]
    fields = [pa.field(f, pa.string()) for f in field_order]
    new_struct = pa.StructArray.from_arrays(arrays, fields=fields)
    new_conv = pa.ListArray.from_arrays(conv_arr.offsets, new_struct)
    idx = pa_table.schema.get_field_index('conversations')
    new_table = pa_table.set_column(idx, 'conversations', new_conv)
    return Dataset(new_table)

vn_rows = []
for record in vietnamese_records:
    img_path = VIETNAMESE_IMAGES_PATH / record['image']
    try:
        image = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f"Warning: cannot open {img_path}: {e}")
        continue

    convs = [normalize_turn(t) for t in record['conversations']]
    pairs = [(convs[i], convs[i+1]) for i in range(0, len(convs) - 1, 2)]

    for idx, (q, a) in enumerate(pairs):
        record_id = record['id'] if len(pairs) == 1 else f"{record['id']}_q{idx}"
        vn_rows.append({
            'id': record_id,
            'image': image,
            'conversations': [q, a],  # chi 1 cap Q&A
        })

print(f"Vietnamese rows sau khi tach cap Q&A: {len(vn_rows)}")

TEST_SIZE = 200
vn_train_rows = vn_rows[:-TEST_SIZE]
vn_test_rows  = vn_rows[-TEST_SIZE:]

vi_vietnamese_train = Dataset.from_list(vn_train_rows)
vi_vietnamese_test  = Dataset.from_list(vn_test_rows)

vi_vietnamese_train = align_conversations_schema(vi_vietnamese_train, vi_chart_dataset['train'])
vi_vietnamese_test  = align_conversations_schema(vi_vietnamese_test,  vi_chart_dataset['test'])

print(f"Vietnamese train: {len(vi_vietnamese_train)} samples")
print(f"Vietnamese test:  {len(vi_vietnamese_test)} samples")

vi_chart_30k = vi_chart_dataset['train'].shuffle(seed=42, keep_in_memory=True).select(range(30000))

merged_train = concatenate_datasets([
    vi_chart_30k,
    vi_vietnamese_train
])
merged_test = concatenate_datasets([
    vi_chart_dataset['test'],
    vi_vietnamese_test
])

vi_chart_dataset['train'] = merged_train
vi_chart_dataset['test']  = merged_test

print("\n=== Final merged dataset ===")
print(vi_chart_dataset)
print(f"Total train samples: {len(vi_chart_dataset['train'])}")
print(f"Total test samples:  {len(vi_chart_dataset['test'])}")


In [ ]:
vi_chart_dataset['train'][0]

In [ ]:
%cd {WORKING / "Vintern"}

In [ ]:
!mkdir -p pretrained

%cd pretrained

!huggingface-cli download --resume-download --local-dir-use-symlinks False 5CD-AI/Vintern-1B-v2 --local-dir Vintern-1B-v2


In [ ]:
import os
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer
import matplotlib.pyplot as plt


In [ ]:
torch.cuda.empty_cache() 

In [ ]:
%cd {WORKING / "Vintern"}

In [ ]:
import os
from tqdm import tqdm

FOLDER_IMAGES = WORKING / "Viet-Chart-VQA-images"
os.makedirs(FOLDER_IMAGES, exist_ok=True)

In [ ]:
for i in tqdm(range(len(vi_chart_dataset['train']))):
    image = vi_chart_dataset['train'][i]['image']
    id = vi_chart_dataset['train'][i]['id']

    image = image.resize((448, 448)) 
    image.save(f"{FOLDER_IMAGES}/{id}.jpg")

In [ ]:
import json
from tqdm import tqdm

def normalize_conversations(conversations):
    role_map = {
        'human': 'human', 'user': 'human',
        'gpt': 'gpt', 'assistant': 'gpt'
    }
    result = []
    for turn in conversations:
        raw_role = turn.get('from') or turn.get('role', '')
        role = role_map.get(str(raw_role).lower(), raw_role)
        value = str(turn.get('content') or turn.get('value', ''))
        result.append({'from': role, 'value': value})
    return result

all_data = []
for item in tqdm(vi_chart_dataset['train']):
    id = item['id']
    image = f"{item['id']}.jpg"
    width = item['image'].width
    height = item['image'].height

    normalized_convs = normalize_conversations(item['conversations'])

    for i in range(0, len(normalized_convs), 2):
        if i + 1 < len(normalized_convs):
            conversations = [
                {
                    "from": "human",
                    "value": normalized_convs[i]['value']
                },
                {
                    "from": "gpt",
                    "value": normalized_convs[i + 1]['value']
                }
            ]

            data = {
                "id": id,
                "image": image,
                "width": width,
                "height": height,
                "conversations": conversations,
            }

            all_data.append(data)

print(len(all_data))
print(json.dumps(all_data[0:5], indent=2, ensure_ascii=False))


In [ ]:
with open(WORKING / 'Vintern/internvl_chat/shell/data/viet-chart-vqa.jsonl', 'w', encoding='utf-8') as f:
    for data in tqdm(all_data):
        f.write(json.dumps(data, ensure_ascii=False) + '\n')

In [ ]:
total_train_samples = len(vi_chart_dataset['train'])
print(f"Total training samples: {total_train_samples}")

metadata_datasets = {
  "vi-chart-vqa": {
    "root": str(WORKING / "Viet-Chart-VQA-images"),
    "annotation": str(WORKING / "Vintern/internvl_chat/shell/data/viet-chart-vqa.jsonl"),
    "data_augment": False,
    "repeat_time": 1,
    "length": total_train_samples
  }
}

with open(WORKING / "Vintern/internvl_chat/shell/data/custom_fintune_datasets.json", 'w', encoding='utf-8') as f:
    json.dump(metadata_datasets, f, ensure_ascii=False, indent=4)

print("Saved custom_fintune_datasets.json")


In [ ]:
!pwd

In [ ]:
!find -name internvl_chat_finetune.py

In [ ]:
OUTPUT_DIR = WORKING / 'work_dirs/internvl_chat_v2_0/Vintern_1B_v2_finetune_lora_viet_chart_vqa'
model_name_or_path = WORKING / "Vintern/pretrained/Vintern-1B-v2"
meta_path = WORKING / "Vintern/internvl_chat/shell/data/custom_fintune_datasets.json"

train_bash_content = f'''#!/bin/bash
set -x
GPUS=${{GPUS:-1}}
BATCH_SIZE=${{BATCH_SIZE:-1}}
PER_DEVICE_BATCH_SIZE=${{PER_DEVICE_BATCH_SIZE:-1}}
GRADIENT_ACC=$((BATCH_SIZE / PER_DEVICE_BATCH_SIZE / GPUS))

export PYTHONPATH="${{PYTHONPATH}}:$(pwd)"
export MASTER_PORT=34229
export TF_CPP_MIN_LOG_LEVEL=3
export LAUNCHER=pytorch

OUTPUT_DIR="{OUTPUT_DIR}"

if [ ! -d "$OUTPUT_DIR" ]; then
  mkdir -p "$OUTPUT_DIR"
fi

torchrun \\
  --nnodes=1 \\
  --node_rank=0 \\
  --master_addr=127.0.0.1 \\
  --nproc_per_node=${{GPUS}} \\
  --master_port=${{MASTER_PORT}} \\
  internvl/train/internvl_chat_finetune.py \\
  --model_name_or_path "{model_name_or_path}" \\
  --conv_style Hermes-2 \\
  --output_dir "${{OUTPUT_DIR}}" \\
  --meta_path "{meta_path}" \\
  --overwrite_output_dir True \\
  --force_image_size 448 \\
  --max_dynamic_patch 6 \\
  --down_sample_ratio 0.5 \\
  --drop_path_rate 0.0 \\
  --freeze_llm True \\
  --freeze_mlp True \\
  --freeze_backbone True \\
  --use_llm_lora 32 \\
  --vision_select_layer -1 \\
  --dataloader_num_workers 4 \\
  --bf16 True \\
  --num_train_epochs 1 \\
  --per_device_train_batch_size ${{PER_DEVICE_BATCH_SIZE}} \\
  --gradient_accumulation_steps ${{GRADIENT_ACC}} \\
  --evaluation_strategy no \\
  --save_strategy steps \\
  --save_steps 500 \\
  --save_total_limit 2 \\
  --learning_rate 4e-5 \\
  --weight_decay 0.01 \\
  --warmup_ratio 0.03 \\
  --lr_scheduler_type cosine \\
  --logging_steps 100 \\
  --max_seq_length 1024 \\
  --do_train True \\
  --grad_checkpoint True \\
  --group_by_length True \\
  --dynamic_image_size True \\
  --use_thumbnail True \\
  --ps_version v2 \\
  --deepspeed zero_stage1_config.json \\
  --report_to tensorboard \\
  2>&1 | tee -a "${{OUTPUT_DIR}}/training_log.txt"
'''

In [ ]:
with open(WORKING / "Vintern/internvl_chat/shell/internvl2.0/2nd_finetune/internvl2_1b_qwen2_0_5b_dynamic_res_2nd_finetune_lora.sh", "w") as text_file:
    text_file.write(train_bash_content)

In [ ]:
import re

trainer_path = '/usr/local/lib/python3.12/dist-packages/transformers/trainer.py'

with open(trainer_path, 'r') as f:
    content = f.read()

old = 'checkpoint_rng_state = torch.load(rng_file)'
new = 'checkpoint_rng_state = torch.load(rng_file, weights_only=False)'

if old in content:
    content = content.replace(old, new)
    with open(trainer_path, 'w') as f:
        f.write(content)
    print('Patched trainer.py successfully')
elif new in content:
    print('trainer.py already patched')
else:
    print('WARNING: Could not find target line in trainer.py')
    for j, line in enumerate(content.splitlines()):
        if 'rng_file' in line and 'torch.load' in line:
            print(f'  Line {j}: {line}')


In [ ]:
if IS_TRAIN: 
    %cd {WORKING / "Vintern/internvl_chat"}
    !sh shell/internvl2.0/2nd_finetune/internvl2_1b_qwen2_0_5b_dynamic_res_2nd_finetune_lora.sh

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/work_dirs_backup', 'zip', '/kaggle/working', 'work_dirs')
print(' Done: /kaggle/working/work_dirs_backup.zip')